# 第3章 GPU 体系结构（下）：片上资源与数据通路 - 操作手册

**Goal**: 理解片上资源（VGPR/SGPR/LDS）、内存层级、合并访存、LDS bank 冲突和 WMMA 矩阵指令。

**Prerequisite**: 已完成第1-2章，理解线程层级和波前执行模型。

**Platform**: 原生 Ubuntu 24.04（推荐）或 WSL2，gfx1201 为叙述基线。

**参考文档**: `docs/part0-intro/chapter3/index.md`

**代码目录**: `code/part0-intro/chapter3/`

## 1. 定位仓库根目录

## 2. 检测 GPU 架构

**Parameter**: 无

**Execution**: 运行 `rocminfo` 检测当前 GPU 架构。

**Expected output**: 输出检测到的架构（gfx1100/gfx1151/gfx1201）。

**Pass criteria**: 成功检测到支持的架构之一。

In [ ]:
# 检测当前 GPU 架构
import subprocess

rocminfo_result = subprocess.run(
    ["rocminfo"],
    capture_output=True,
    text=True,
    check=True
)

# 从 rocminfo 输出中提取架构
arch = "gfx1201"  # 默认值（gfx1201 为叙述基线）
if "gfx1100" in rocminfo_result.stdout:
    arch = "gfx1100"
elif "gfx1151" in rocminfo_result.stdout:
    arch = "gfx1151"
elif "gfx1201" in rocminfo_result.stdout:
    arch = "gfx1201"
else:
    raise RuntimeError("未检测到支持的架构 (gfx1100/gfx1151/gfx1201)")

print(f"检测到架构: {arch}")


In [ ]:
import pathlib
import subprocess

def find_repo_root():
    current = pathlib.Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").is_dir()
            and (candidate / "docs").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError("无法定位仓库根目录")

REPO_ROOT = find_repo_root()
print(f"仓库根目录: {REPO_ROOT}")

## 3. 全局内存访问实验

**Parameter**:
- 实现: stride-1（合并访存）vs stride-17 vs stride-257（非合并访存）
- 数据规模: 16M FP32 元素
- warmup: 10 次
- repeat: 50 次

**Execution**: 编译并运行 `global_memory_access.hip`，对比不同 stride 的性能。

**Expected output**: 
- 各 stride 的执行时间和逻辑有效带宽
- stride-1 应有最高带宽（合并访存）
- stride 增大时带宽显著下降

**Pass criteria**: 程序成功编译运行，输出包含三种 stride 的性能数据。

**Platform-specific commands**:
- gfx1100: `hipcc --offload-arch=gfx1100 -O3 -std=c++17 global_memory_access.hip -o global_memory_access`
- gfx1151: `hipcc --offload-arch=gfx1151 -O3 -std=c++17 global_memory_access.hip -o global_memory_access`
- gfx1201: `hipcc --offload-arch=gfx1201 -O3 -std=c++17 global_memory_access.hip -o global_memory_access`

In [ ]:
chapter3_dir = REPO_ROOT / "code/part0-intro/chapter3"
global_mem_hip = chapter3_dir / "global_memory_access.hip"
global_mem_bin = chapter3_dir / "global_memory_access"

# 编译（以 gfx1201 为基线）
compile_result = subprocess.run(
    ["hipcc", f"--offload-arch={arch}", "-O3", "-std=c++17",
     str(global_mem_hip), "-o", str(global_mem_bin)],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter3_dir
)
print("编译成功")

# 运行实验
run_result = subprocess.run(
    [str(global_mem_bin), "--implementation", "all", "--size", "16777216",
     "--warmup", "10", "--repeat", "50"],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter3_dir
)
print("\n实验结果:")
print(run_result.stdout)

print("\n解释:")
print("- stride-1: 相邻 lanes 读取相邻地址，合并访存，带宽最高")
print("- stride-17/257: 地址离散，缓存利用率低，带宽显著下降")
print("- 合并访存是 GPU 性能优化的关键")

## 4. LDS Bank 冲突实验

**Parameter**:
- 实现: stride-1（无冲突）vs stride-32（最大冲突）vs stride-33（错开）
- 数据规模: 默认
- warmup: 10 次
- repeat: 50 次

**Execution**: 编译并运行 `lds_bank_conflict.hip`，对比不同 stride 的 LDS 访问性能。

**Expected output**:
- 各 stride 的执行时间
- stride-32 应有最差性能（bank 冲突）
- stride-1 和 stride-33 性能相近（无冲突）

**Pass criteria**: 程序成功编译运行，输出包含三种 stride 的性能数据。

**Platform-specific commands**:
- gfx1100: `hipcc --offload-arch=gfx1100 -O3 -std=c++17 lds_bank_conflict.hip -o lds_bank_conflict`
- gfx1151: `hipcc --offload-arch=gfx1151 -O3 -std=c++17 lds_bank_conflict.hip -o lds_bank_conflict`
- gfx1201: `hipcc --offload-arch=gfx1201 -O3 -std=c++17 lds_bank_conflict.hip -o lds_bank_conflict`

In [ ]:
lds_bank_hip = chapter3_dir / "lds_bank_conflict.hip"
lds_bank_bin = chapter3_dir / "lds_bank_conflict"

# 编译（以 gfx1201 为基线）
compile_result = subprocess.run(
    ["hipcc", f"--offload-arch={arch}", "-O3", "-std=c++17",
     str(lds_bank_hip), "-o", str(lds_bank_bin)],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter3_dir
)
print("编译成功")

# 运行实验
run_result = subprocess.run(
    [str(lds_bank_bin), "--implementation", "all",
     "--warmup", "10", "--repeat", "50"],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter3_dir
)
print("\n实验结果:")
print(run_result.stdout)

print("\n解释:")
print("- LDS 内部分为多个 bank，可并行访问")
print("- stride-1: 每个 lane 访问不同 bank，无冲突")
print("- stride-32: 多个 lanes 访问同一 bank，产生冲突，性能下降")
print("- stride-33: 错开 bank 分布，避免冲突")

## 5. WMMA 矩阵指令实验

**Parameter**:
- 架构自动选择:
  - gfx1100/gfx1151 → rdna3_wmma.hip
  - gfx1201 → rdna4_wmma.hip
- 矩阵规模: 默认
- warmup: 5 次
- repeat: 20 次

**Execution**: 
1. 根据 GPU 架构选择对应的 WMMA 实现
2. 编译并运行，对比 VALU 和 WMMA 的性能

**Expected output**:
- VALU 实现的执行时间和 TFLOPS
- WMMA 实现的执行时间和 TFLOPS
- WMMA 应显著快于 VALU

**Pass criteria**: 
- 程序成功编译运行
- WMMA 正确性验证通过（correctness before performance）
- 输出包含 VALU 和 WMMA 的性能对比

**Platform-specific commands (three offload arches)**:
- gfx1100: `hipcc --offload-arch=gfx1100 -O3 -std=c++17 rdna3_wmma.hip -o wmma_test`
- gfx1151: `hipcc --offload-arch=gfx1151 -O3 -std=c++17 rdna3_wmma.hip -o wmma_test`
- gfx1201: `hipcc --offload-arch=gfx1201 -O3 -std=c++17 rdna4_wmma.hip -o wmma_test`

In [ ]:
# WMMA 源文件选择（基于架构）
# gfx1100/gfx1151 使用 rdna3_wmma.hip
# gfx1201 使用 rdna4_wmma.hip
if arch in ["gfx1100", "gfx1151"]:
    wmma_source = "rdna3_wmma.hip"
elif arch == "gfx1201":
    wmma_source = "rdna4_wmma.hip"
else:
    raise RuntimeError(f"不支持的架构: {arch}")

print(f"使用 WMMA 源文件: {wmma_source}")

wmma_hip = chapter3_dir / wmma_source
wmma_bin = chapter3_dir / "wmma_test"

compile_result = subprocess.run(
    ["hipcc", f"--offload-arch={arch}", "-O3", "-std=c++17",
     str(wmma_hip), "-o", str(wmma_bin)],
    capture_output=True,
    text=True,
    check=True,
    cwd=chapter3_dir
)
print(f"编译成功: {wmma_bin}")


## 总结

本章完成了 GPU 片上资源和数据通路的核心实验：
1. ✓ 理解全局内存访问模式对性能的影响（合并访存）
2. ✓ 观察 LDS bank 冲突现象
3. ✓ 对比 VALU 和 WMMA 的性能差异
4. ✓ 建立内存层级和片上资源的直觉

这些概念是后续算子优化的直接基础。